In [3]:
!pip install mxnet

  Using cached mxnet-1.7.0.post2-py2.py3-none-win_amd64.whl.metadata (402 bytes)
  Using cached numpy-1.16.6.zip (5.1 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'error'


  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [30 lines of output]
      Running from numpy source directory.
      <string>:394: UserWarning: Unrecognized setuptools command, proceeding with generating Cython sources and expanding templates
      C:\Users\yuyue\AppData\Local\Temp\pip-install-2v5o6810\numpy_50eba3d478fa4fadbea13ee39132f304\numpy\distutils\misc_util.py:476: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
        return is_string(s) and ('*' in s or '?' is s)
      Traceback (most recent call last):
        File "C:\Users\yuyue\AppData\Local\Programs\Python\Python313\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 389, in <module>
          main()
          ~~~~^^
        File "C:\Users\yuyue\AppData\Local\Programs\Python\Python313\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 373, in main
          json_out["ret

In [2]:
import mxnet as mx
import gluonnlp as nlp
import numpy as np
from tqdm import tqdm

def embed_texts_gluonnlp(
    texts,
    model_name="bert_12_768_12",
    dataset="book_corpus_wiki_en_uncased",
    batch_size=32,
    ctx=None,
    layers=(1, 6, 12)
):
    """
    Compute mean-pooled embeddings from GluonNLP BERT, averaging over given layers.
    
    Parameters:
        texts: list[str] - input texts
        model_name: str - gluonnlp model (bert_12_768_12)
        dataset: str - which pretrained weights to use
        batch_size: int - batching for speed
        ctx: mx.Context - e.g. mx.gpu(0) or mx.cpu()
        layers: tuple[int] - layers to extract (paper uses 1,6,12)

    Returns:
        np.ndarray [num_texts, hidden_dim] - mean-pooled, layer-averaged embeddings
    """
    ctx = ctx or mx.cpu()
    
    # Load model + vocab
    bert, vocab = nlp.model.get_model(
        model_name,
        dataset_name=dataset,
        pretrained=True,
        use_pooler=True,
        use_decoder=False,
        use_classifier=False,
        ctx=ctx
    )

    tokenizer = nlp.data.BERTTokenizer(vocab, lower=True)
    transform = nlp.data.BERTSentenceTransform(
        tokenizer, max_seq_length=128, pad=True, pair=False
    )

    dataloader = []
    for txt in texts:
        dataloader.append(transform([txt]))

    input_ids = np.stack([x[0] for x in dataloader])
    valid_len = np.array([x[1] for x in dataloader])
    segment_ids = np.stack([x[2] for x in dataloader])

    # Convert to MXNet NDArrays
    input_ids_nd = mx.nd.array(input_ids, ctx=ctx)
    segment_ids_nd = mx.nd.array(segment_ids, ctx=ctx)
    valid_len_nd = mx.nd.array(valid_len, ctx=ctx)

    all_embs = []
    for i in tqdm(range(0, len(texts), batch_size)):
        ids_batch = input_ids_nd[i:i+batch_size]
        seg_batch = segment_ids_nd[i:i+batch_size]
        valid_batch = valid_len_nd[i:i+batch_size]

        # Get all layers' hidden states
        _, _, additional_outputs = bert(ids_batch, seg_batch, valid_length=valid_batch)
        # additional_outputs[0] = list of hidden states (length num_layers+1)
        hidden_states = additional_outputs[0]

        # Pick selected layers and mean-pool over tokens
        selected = [hidden_states[L] for L in layers]
        layer_embs = []
        for layer_tensor in selected:
            # layer_tensor: (batch, seq_len, hidden_dim)
            mask = mx.nd.arange(layer_tensor.shape[1], ctx=ctx).expand_dims(0) < valid_batch.expand_dims(1)
            mask = mask.astype(layer_tensor.dtype)
            summed = (layer_tensor * mask.expand_dims(-1)).sum(axis=1)
            mean_pooled = summed / valid_batch.expand_dims(1)
            layer_embs.append(mean_pooled)

        # Average across layers
        layer_avg = sum(layer_embs) / len(layer_embs)
        all_embs.append(layer_avg.asnumpy())

    return np.vstack(all_embs)


ModuleNotFoundError: No module named 'mxnet'